# 24C — competition-only 96×384 MRI cache, shard 0 of 4 (CPU)

This is a **real cache build**, not another benchmark. It processes only the weak/report-only training studies assigned to shard 0 by sorted StudyInstanceUID round-robin. It uses the same five-slot, 96-slice, 140 mm crop, 384×384 uint8 representation measured in notebook 24B. No OAI or other MRI dataset is read.

Attach **only** RSNA Knee Abnormality Detection competition data. Settings: **Accelerator None**, **Internet Off**. Save & Run All. The output contains chunked ZIP archives, a CSV manifest, and `24C_shard0_receipt.json`. Send only the small receipt and manifest first; do not download the large ZIPs. Do **not** run shards 1–3 until shard 0 is audited.

This cache is for a *new competition-only model*. It is not claimed byte-identical to the unpublished 0.954 author's training corpus. The eventual training and inference notebooks must use this exact preprocessing. This notebook does not train or submit. Expected runtime is uncertain; the six-study 24B benchmark is not a reliable full-run ETA.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import monotonic
import gc, hashlib, io, json, os, shutil, zipfile
import numpy as np
import pandas as pd

T0 = monotonic()
SHARD_ID = 0
N_SHARDS = 4
CHUNK_STUDIES = 100
MAX_OUTPUT_BYTES = 16 * 2**30
IMG = 384
CROP_MM = 140.0
SLOTS = [('Sagittal', 1, 26), ('Sagittal', 0, 22), ('Coronal', 1, 18), ('Coronal', 0, 12), ('Axial', -1, 18)]
WORK = Path('/kaggle/working')
OUTDIR = WORK / f'rsna_24c_shard{SHARD_ID}'
RECEIPT = WORK / f'24C_shard{SHARD_ID}_receipt.json'
report = {'experiment': '24C_competition_only_cache_shard', 'time_utc': datetime.now(timezone.utc).isoformat(), 'shard_id': SHARD_ID, 'n_shards': N_SHARDS, 'chunk_studies': CHUNK_STUDIES, 'max_output_bytes': MAX_OUTPUT_BYTES, 'training_started': False, 'cuda_used': False, 'errors': [], 'warnings': []}
root = next((p for p in (Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'), Path('/kaggle/input/rsna-knee-abnormality-detection')) if (p / 'train.csv').is_file()), None)
report['competition_root'] = str(root) if root else None
try:
    import cv2, pydicom
    from pydicom.pixel_data_handlers.util import apply_modality_lut
    dependencies_ok = True
except Exception as e:
    dependencies_ok = False
    report['errors'].append('Dependency error: ' + type(e).__name__ + ': ' + str(e)[:200])
print('CPU shard builder', SHARD_ID, '/', N_SHARDS, '| root:', root, '| dependencies:', dependencies_ok)


In [ ]:
def header_key(path):
    try:
        d = pydicom.dcmread(str(path), stop_before_pixels=True)
        iop = getattr(d, 'ImageOrientationPatient', None)
        ipp = getattr(d, 'ImagePositionPatient', None)
        if iop is not None and ipp is not None and len(iop) == 6:
            r = np.asarray(iop[:3], dtype=float); c = np.asarray(iop[3:], dtype=float)
            pos = float(np.dot(np.asarray(ipp, dtype=float), np.cross(r, c)))
        else:
            pos = float(getattr(d, 'InstanceNumber', 0) or 0)
        spacing = getattr(d, 'PixelSpacing', None)
        spacing = float(spacing[0]) if spacing is not None else 0.5
        return pos, path, spacing
    except Exception:
        return 0.0, path, 0.5

def decode(path):
    d = pydicom.dcmread(str(path))
    a = np.asarray(apply_modality_lut(d.pixel_array, d), dtype=np.float32)
    if a.ndim != 2: raise ValueError('Expected 2-D DICOM pixel array')
    if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1': a = a.max() - a
    return a

def build_one(uid, rows):
    volume = np.zeros((96, IMG, IMG), dtype=np.uint8)
    used, offset, decoded, failed = set(), 0, 0, 0
    for plane, fluid, k in SLOTS:
        candidates = [r for r in rows if str(r.get('Anatomical_Plane', '')) == plane and str(r.get('SeriesInstanceUID', '')) not in used]
        preferred = [r for r in candidates if int(r.get('Fluid_Sensitive', 0) or 0) == fluid] if fluid >= 0 else []
        chosen = preferred[0] if preferred else (candidates[0] if candidates else None)
        if chosen is None:
            offset += k; continue
        series_uid = str(chosen['SeriesInstanceUID'])
        used.add(series_uid)
        files = list((root / 'train_series' / uid / series_uid).glob('*.dcm'))
        if not files:
            offset += k; failed += k; continue
        ordered = sorted((header_key(p) for p in files), key=lambda x: x[0])
        n = len(ordered); lo = int(n * 0.02); hi = max(lo, int(n * 0.98) - 1)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else np.zeros(k, dtype=int)
        chosen_files = [ordered[min(int(i), n - 1)] for i in picks]
        arrays = []
        for _, path, spacing in chosen_files:
            try:
                arrays.append((decode(path), spacing)); decoded += 1
            except Exception:
                arrays.append((None, spacing)); failed += 1
        valid = [a for a, _ in arrays if a is not None]
        if valid:
            values = np.concatenate([a.ravel() for a in valid])
            qlo, qhi = np.percentile(values, [2, 98])
            del values
        else:
            qlo, qhi = 0.0, 1.0
        for j, (a, spacing) in enumerate(arrays):
            if a is None: continue
            a = np.clip((a - qlo) / (qhi - qlo + 1e-6), 0, 1)
            h, w = a.shape; box = min(int(round(CROP_MM / max(spacing, 1e-3))), h, w)
            y = (h - box) // 2; x = (w - box) // 2
            volume[offset + j] = (cv2.resize(a[y:y+box, x:x+box], (IMG, IMG), interpolation=cv2.INTER_AREA) * 255).astype(np.uint8)
        offset += k
        del arrays, valid
    mask = np.any(volume != 0, axis=(1, 2)).astype(np.uint8)
    return volume, mask, decoded, failed


In [ ]:
if OUTDIR.exists() and any(OUTDIR.iterdir()):
    report['status'] = 'OUTPUT_EXISTS'
    report['errors'].append('Existing shard output is present; use a fresh Kaggle run. No files were overwritten.')
elif root is None:
    report['status'] = 'NEEDS_COMPETITION'
elif not dependencies_ok:
    report['status'] = 'NEEDS_DEPENDENCIES'
else:
    train_path = root / 'train.csv'
    series_path = root / 'train_series.csv'
    train = pd.read_csv(train_path, dtype={'StudyInstanceUID': str})
    series = pd.read_csv(series_path, dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    required = {'StudyInstanceUID', 'SeriesInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive'}
    if not required.issubset(series.columns):
        report['status'] = 'MISSING_SERIES_COLUMNS'
        report['errors'].append('Found columns: ' + str(list(series.columns)))
    else:
        labels = [c for c in train.columns if c not in ('StudyInstanceUID', 'Report')]
        weak_ids = sorted(set(train.loc[train[labels].isna().all(axis=1), 'StudyInstanceUID'].astype(str)))
        group = {str(k): g.to_dict('records') for k, g in series.groupby('StudyInstanceUID', sort=False) if str(k) in weak_ids}
        missing_series = sorted(set(weak_ids) - set(group))
        assigned = weak_ids[SHARD_ID::N_SHARDS]
        report['weak_studies'] = len(weak_ids)
        report['assigned_studies'] = len(assigned)
        report['missing_series_count'] = len(missing_series)
        report['assigned_uid_sha256'] = hashlib.sha256('\n'.join(assigned).encode()).hexdigest()
        for name, path in (('train_csv_sha256', train_path), ('train_series_csv_sha256', series_path)):
            with path.open('rb') as f:
                report[name] = hashlib.file_digest(f, 'sha256').hexdigest()
        if len(weak_ids) != 4349 or missing_series:
            report['status'] = 'INPUT_IDENTITY_MISMATCH'
            report['errors'].append('Expected 4349 weak IDs with series; found ' + str(len(weak_ids)) + ', missing series ' + str(len(missing_series)))
        else:
            OUTDIR.mkdir(parents=True, exist_ok=True)
            manifest_path = OUTDIR / 'manifest.csv'
            rows, built_bytes, writer, chunk_number = [], 0, None, 0
            stop_reason = None
            try:
                for index, uid in enumerate(assigned):
                    t = monotonic()
                    try:
                        volume, mask, decoded, failed = build_one(uid, group[uid])
                        valid = int(mask.sum())
                        if valid < 40:
                            raise ValueError('Only ' + str(valid) + ' valid slices; inspect this study')
                        buffer = io.BytesIO()
                        np.savez_compressed(buffer, images=volume, mask=mask)
                        payload = buffer.getvalue()
                        del buffer, volume, mask
                        if built_bytes + len(payload) + 1_000_000 > MAX_OUTPUT_BYTES:
                            stop_reason = 'Output guard at 16 GiB before study ' + uid
                            break
                        if writer is None:
                            chunk_number = len(rows) // CHUNK_STUDIES
                            archive_name = f'chunk_{chunk_number:02d}.zip'
                            writer = zipfile.ZipFile(OUTDIR / archive_name, 'w', allowZip64=True, compression=zipfile.ZIP_STORED)
                        entry_name = 'study_' + uid + '.npz'
                        writer.writestr(entry_name, payload)
                        rows.append({'StudyInstanceUID': uid, 'archive': archive_name, 'entry': entry_name, 'npz_bytes': len(payload), 'npz_sha256': hashlib.sha256(payload).hexdigest(), 'valid_slices': valid, 'decoded_slices': decoded, 'failed_slices': failed})
                        built_bytes += len(payload)
                        del payload
                        if len(rows) % CHUNK_STUDIES == 0:
                            writer.close(); writer = None
                            pd.DataFrame(rows).to_csv(manifest_path, index=False)
                            print('completed', len(rows), '/', len(assigned), 'studies | GiB', round(built_bytes / 2**30, 2), '| minutes', round((monotonic()-T0)/60, 1), flush=True)
                    except Exception as e:
                        report['errors'].append({'uid': uid, 'error': type(e).__name__ + ': ' + str(e)[:250]})
                        if len(report['errors']) >= 10:
                            stop_reason = 'Ten study errors; stopped to prevent further bad output'
                            break
                    finally:
                        gc.collect()
            finally:
                if writer is not None:
                    writer.close()
                pd.DataFrame(rows, columns=['StudyInstanceUID','archive','entry','npz_bytes','npz_sha256','valid_slices','decoded_slices','failed_slices']).to_csv(manifest_path, index=False)
            report['built_studies'] = len(rows)
            report['built_bytes_gib'] = round(built_bytes / 2**30, 3)
            report['zip_files'] = [{'name': p.name, 'bytes': p.stat().st_size} for p in sorted(OUTDIR.glob('chunk_*.zip'))]
            report['actual_output_gib'] = round(sum(x['bytes'] for x in report['zip_files']) / 2**30, 3)
            expected_entries = {}
            for row in rows: expected_entries.setdefault(row['archive'], set()).add(row['entry'])
            inventory_ok = True
            for archive_name, expected_names in expected_entries.items():
                try:
                    with zipfile.ZipFile(OUTDIR / archive_name) as z:
                        if set(z.namelist()) != expected_names: inventory_ok = False
                except Exception as e:
                    inventory_ok = False
                    report['errors'].append({'archive': archive_name, 'error': type(e).__name__ + ': ' + str(e)[:200]})
            report['zip_inventory_ok'] = inventory_ok
            report['manifest'] = str(manifest_path)
            report['status'] = 'COMPLETE' if len(rows) == len(assigned) and not report['errors'] and inventory_ok else 'INCOMPLETE'
            if stop_reason: report['warnings'].append(stop_reason)
report['elapsed_seconds'] = round(monotonic() - T0, 2)
RECEIPT.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print('SHARD STATUS:', report['status'], '| built', report.get('built_studies', 0), '/', report.get('assigned_studies', 0), '| minutes', round(report['elapsed_seconds']/60, 1))
print('Receipt:', RECEIPT)
